In [1]:
import numpy as np

# =========================
# PRECISION CONTROL
# =========================
# Switch between np.float64 and np.longdouble here.
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)

# =========================
# USER SETTINGS
# =========================
R = DT("8.314")
T = DT("1000.0")

# MQMQA entropy exponents (choose either)
phi = 1.0   # original MQMQA often uses phi=1
psi = 1.0   # original MQMQA often uses psi=1
# (if you want "updated" style exponents, you can try phi=0.75, psi=0.5)

# Turn blocks on/off (useful for unit-testing S1, S2, S3 separately)
USE_S1 = True
USE_S2 = True
USE_S3 = True

# FD step sizes to sweep
H_LIST = [DT("1e-2"), DT("1e-3"), DT("1e-4"), DT("1e-5"), DT("1e-6")]

# Base quadruplet moles (must all be > 0, and remain > 0 under +/-h)
# 9 quadruplets for 2 cations x 2 anions:
# [AA/XX, AA/XY, AA/YY, AB/XX, AB/XY, AB/YY, BB/XX, BB/XY, BB/YY]
n0 = np.array([DT("1.00"), DT("0.90"), DT("0.80"),
               DT("0.70"), DT("0.60"), DT("0.50"),
               DT("0.40"), DT("0.30"), DT("0.20")], dtype=DT)

# =========================
# TOY MQMQA INDEX SETUP
# =========================
cations = ["A", "B"]
anions  = ["X", "Y"]

def delta(a, b):
    return DT("1.0") if a == b else DT("0.0")

# Build quadruplets r=(i,j,k,l) with i<=j and k<=l
quad = []
for i_idx, i in enumerate(cations):
    for j in cations[i_idx:]:
        for k_idx, k in enumerate(anions):
            for l in anions[k_idx:]:
                quad.append((i, j, k, l))

M = len(quad)
quad_names = [f"{i}{j}/{k}{l}" for (i, j, k, l) in quad]
ONES = np.ones(M, dtype=DT)

# =========================
# LINEAR WEIGHTS (toy but consistent)
# =========================
# Site moles on 1st sublattice (cations): n_i = sum_r n_r * (δ_ai+δ_bi)/2
w_site1 = {
    i: np.array([(delta(a, i) + delta(b, i)) / DT("2.0") for (a, b, x, y) in quad], dtype=DT)
    for i in cations
}

# Site moles on 2nd sublattice (anions): n_k = sum_r n_r * (δ_xk+δ_yk)/2
w_site2 = {
    k: np.array([(delta(x, k) + delta(y, k)) / DT("2.0") for (a, b, x, y) in quad], dtype=DT)
    for k in anions
}

# Pair moles: n_{i/k} = sum_r n_r * ((δ_ai+δ_bi)(δ_xk+δ_yk))/4
w_pair = {
    (i, k): np.array([
        ((delta(a, i) + delta(b, i)) * (delta(x, k) + delta(y, k))) / DT("4.0")
        for (a, b, x, y) in quad
    ], dtype=DT)
    for i in cations for k in anions
}

# Row sums A_i = sum_k n_{i/k}
w_row = {i: sum(w_pair[(i, k)] for k in anions) for i in cations}

# Col sums B_k = sum_i n_{i/k}
w_col = {k: sum(w_pair[(i, k)] for i in cations) for k in anions}

# Total pair moles Npair = sum_{i,k} n_{i/k}
w_Npair = sum(w_pair[(i, k)] for i in cations for k in anions)  # equals ONES in this toy


# =========================
# HELPER: log-derivatives for linear forms & ratios
# =========================
def ln_linear_derivs(w, n):
    """
    L = w·n (linear). Return:
      ln(L),  (ln L)_{,p} as vector length M,  (ln L)_{,pq} as MxM matrix.
    """
    n = asDT(n)
    L = np.dot(w, n)
    if L <= 0:
        raise ValueError("Encountered nonpositive linear form inside log.")
    ln_val = np.log(L)
    ln_p = w / L
    ln_pq = -np.outer(w, w) / (L * L)
    return ln_val, ln_p, ln_pq


def ln_ratio_derivs(wA, wB, n):
    """
    X = A/B with A=wA·n, B=wB·n. Return derivatives of ln(X)=ln(A)-ln(B).
    """
    n = asDT(n)
    A = np.dot(wA, n)
    B = np.dot(wB, n)
    if A <= 0 or B <= 0:
        raise ValueError("Encountered nonpositive ratio parts inside log.")
    ln_val = np.log(A) - np.log(B)
    ln_p = wA / A - wB / B
    ln_pq = -np.outer(wA, wA) / (A * A) + np.outer(wB, wB) / (B * B)
    return ln_val, ln_p, ln_pq


def ln_Xquad_derivs(r, n):
    """
    X_r = n_r / Nquad, Nquad=sum(n). Return derivatives of ln X_r.
    """
    n = asDT(n)
    n_r = n[r]
    N = np.sum(n, dtype=DT)
    if n_r <= 0 or N <= 0:
        raise ValueError("n_r or Nquad nonpositive.")
    e = np.zeros(M, dtype=DT)
    e[r] = DT("1.0")
    ln_val = np.log(n_r) - np.log(N)
    ln_p = e / n_r - ONES / N
    ln_pq = -np.outer(e, e) / (n_r * n_r) + np.outer(ONES, ONES) / (N * N)
    return ln_val, ln_p, ln_pq


# =========================
# STATE + G_id (Eq.16-style)
# =========================
def compute_state(n):
    n = asDT(n)
    if np.any(n <= 0):
        raise ValueError("All quadruplet moles must be > 0 (logs).")

    Nquad = np.sum(n, dtype=DT)

    # site moles and site fractions
    n_site1 = {i: np.dot(w_site1[i], n) for i in cations}
    n_site2 = {k: np.dot(w_site2[k], n) for k in anions}
    N1 = sum(n_site1.values())
    N2 = sum(n_site2.values())
    X_site1 = {i: n_site1[i] / N1 for i in cations}
    X_site2 = {k: n_site2[k] / N2 for k in anions}

    # pair moles and pair fractions
    n_pair = {(i, k): np.dot(w_pair[(i, k)], n) for i in cations for k in anions}
    Npair = sum(n_pair.values())
    X_pair = {(i, k): n_pair[(i, k)] / Npair for i in cations for k in anions}

    # row/col sums and F factors
    A_row = {i: sum(n_pair[(i, k)] for k in anions) for i in cations}
    B_col = {k: sum(n_pair[(i, k)] for i in cations) for k in anions}
    F_i = {i: A_row[i] / Npair for i in cations}
    F_k = {k: B_col[k] / Npair for k in anions}

    # quadruplet fractions
    X_quad = n / Nquad

    # site-equivalent fractions used in the S3 Y-block (single-sublattice):
    # Y_i^(1st)=sum_r X_r*(δ_ai+δ_bi)/2 = n_i / Nquad in this toy
    Y1 = {i: n_site1[i] / Nquad for i in cations}
    Y2 = {k: n_site2[k] / Nquad for k in anions}

    return dict(
        Nquad=Nquad, Npair=Npair, N1=N1, N2=N2,
        n_site1=n_site1, n_site2=n_site2,
        X_site1=X_site1, X_site2=X_site2,
        n_pair=n_pair, X_pair=X_pair,
        A_row=A_row, B_col=B_col, F_i=F_i, F_k=F_k,
        X_quad=X_quad, Y1=Y1, Y2=Y2
    )


def G_id(n):
    """
    Toy MQMQA ideal term:
      G^id = R T [ S1 + S2 + S3 ]
    where:
      S1 = sum_sites n ln X
      S2 = sum_pairs n_{i/k} ln( X_{i/k} / (F_i F_k) )
      S3 = sum_quad  n_r ln u_r, with:
           ln u_r = ln X_r - ln C_r - phi*sum_4pairs ln X_pair + psi*sum_4Y ln Y
    """
    st = compute_state(n)

    S1 = DT("0.0")
    if USE_S1:
        for i in cations:
            ni = st["n_site1"][i]
            Xi = st["X_site1"][i]
            S1 += ni * np.log(Xi)
        for k in anions:
            nk = st["n_site2"][k]
            Xk = st["X_site2"][k]
            S1 += nk * np.log(Xk)

    S2 = DT("0.0")
    if USE_S2:
        Npair = st["Npair"]
        for i in cations:
            Ai = st["A_row"][i]
            for k in anions:
                nik = st["n_pair"][(i, k)]
                Bk = st["B_col"][k]
                # ln( X_{i/k} / (F_i F_k) ) = ln(n_{i/k}) + ln(Npair) - ln(Ai) - ln(Bk)
                S2 += nik * (np.log(nik) + np.log(Npair) - np.log(Ai) - np.log(Bk))

    S3 = DT("0.0")
    if USE_S3:
        Nquad = st["Nquad"]
        for r, (i, j, k, l) in enumerate(quad):
            nr = n[r]
            Xr = nr / Nquad
            C = (DT("2.0") - (DT("1.0") if i == j else DT("0.0"))) * (DT("2.0") - (DT("1.0") if k == l else DT("0.0")))
            ln_u = np.log(Xr) - np.log(C)

            # -phi * ln of 4 pair fractions
            pairs = [(i, k), (i, l), (j, k), (j, l)]
            ln_u -= phi * sum(np.log(st["X_pair"][pk]) for pk in pairs)

            # +psi * ln of 4 Y factors (2 first-sublattice + 2 second-sublattice)
            ln_u += psi * (
                np.log(st["Y1"][i]) + np.log(st["Y1"][j]) +
                np.log(st["Y2"][k]) + np.log(st["Y2"][l])
            )

            S3 += nr * ln_u

    return R * T * (S1 + S2 + S3)


# =========================
# ANALYTIC HESSIAN H_id
# =========================
def H_id_analytic(n):
    n = asDT(n)
    compute_state(n)  # just to assert positivity/log-safety

    H = np.zeros((M, M), dtype=DT)

    # Precompute ln Npair derivatives (Npair is linear in this toy)
    _, lnNpair_p, lnNpair_pq = ln_linear_derivs(w_Npair, n)

    # Precompute ln row/col derivatives
    lnRow = {}
    for i in cations:
        _, lp, lpq = ln_linear_derivs(w_row[i], n)
        lnRow[i] = (lp, lpq)

    lnCol = {}
    for k in anions:
        _, lp, lpq = ln_linear_derivs(w_col[k], n)
        lnCol[k] = (lp, lpq)

    # Precompute ln pair-mole derivatives ln(n_{i/k})
    lnPair = {}
    for i in cations:
        for k in anions:
            _, lp, lpq = ln_linear_derivs(w_pair[(i, k)], n)
            lnPair[(i, k)] = (lp, lpq)

    # ---------- S1 ----------
    if USE_S1:
        # first sublattice: f = n_i ln(n_i/N1), with n_i linear
        for i in cations:
            wN = w_site1[i]
            N = np.dot(wN, n)
            _, ln_p, ln_pq = ln_ratio_derivs(w_site1[i], ONES, n)  # ln(n_i/N1), and N1=ONES·n here
            H += np.outer(wN, ln_p) + np.outer(ln_p, wN) + N * ln_pq

        # second sublattice: f = n_k ln(n_k/N2)
        for k in anions:
            wN = w_site2[k]
            N = np.dot(wN, n)
            _, ln_p, ln_pq = ln_ratio_derivs(w_site2[k], ONES, n)
            H += np.outer(wN, ln_p) + np.outer(ln_p, wN) + N * ln_pq

    # ---------- S2 ----------
    if USE_S2:
        for i in cations:
            for k in anions:
                wN = w_pair[(i, k)]
                N = np.dot(wN, n)  # n_{i/k}

                # ln u_{i/k} = ln n_{i/k} + ln Npair - ln A_i - ln B_k
                ln_u_p  = lnPair[(i, k)][0] + lnNpair_p  - lnRow[i][0] - lnCol[k][0]
                ln_u_pq = lnPair[(i, k)][1] + lnNpair_pq - lnRow[i][1] - lnCol[k][1]

                H += np.outer(wN, ln_u_p) + np.outer(ln_u_p, wN) + N * ln_u_pq

    # ---------- S3 ----------
    if USE_S3:
        # Precompute ln X_pair derivatives: ln X_{i/k} = ln n_{i/k} - ln Npair
        lnXpair = {}
        lnXpair_pq = {}
        for i in cations:
            for k in anions:
                lnXpair[(i, k)]    = lnPair[(i, k)][0] - lnNpair_p
                lnXpair_pq[(i, k)] = lnPair[(i, k)][1] - lnNpair_pq

        # Precompute ln Y^(1st), ln Y^(2nd) derivatives as ratios of linear forms:
        lnY1 = {}
        lnY1_pq = {}
        for i in cations:
            _, lp, lpq = ln_ratio_derivs(w_site1[i], ONES, n)  # ln(n_i/Nquad)
            lnY1[i] = lp
            lnY1_pq[i] = lpq

        lnY2 = {}
        lnY2_pq = {}
        for k in anions:
            _, lp, lpq = ln_ratio_derivs(w_site2[k], ONES, n)  # ln(n_k/Nquad)
            lnY2[k] = lp
            lnY2_pq[k] = lpq

        for r, (i, j, k, l) in enumerate(quad):
            nr = n[r]

            # ln X_r derivatives
            _, lnXr_p, lnXr_pq = ln_Xquad_derivs(r, n)

            # ln u_r derivatives = ln X_r - phi*sum4 ln X_pair + psi*sum4 ln Y
            ln_u_p  = lnXr_p.copy()
            ln_u_pq = lnXr_pq.copy()

            pairs = [(i, k), (i, l), (j, k), (j, l)]
            for pk in pairs:
                ln_u_p  -= phi * lnXpair[pk]
                ln_u_pq -= phi * lnXpair_pq[pk]

            ln_u_p  += psi * (lnY1[i] + lnY1[j] + lnY2[k] + lnY2[l])
            ln_u_pq += psi * (lnY1_pq[i] + lnY1_pq[j] + lnY2_pq[k] + lnY2_pq[l])

            # Hessian of n_r ln u_r:
            # H_pq += δ_{rp}(ln u_r)_q + δ_{rq}(ln u_r)_p + n_r (ln u_r)_{pq}
            H[r, :] += ln_u_p
            H[:, r] += ln_u_p
            H += nr * ln_u_pq

    return R * T * H


# =========================
# FINITE DIFFERENCE HESSIAN
# =========================
def H_id_fd(n0, h):
    n0 = asDT(n0)
    h = DT(h)
    if np.min(n0) <= h:
        raise ValueError("h too large: n0-h must remain positive for all components.")

    H = np.zeros((M, M), dtype=DT)
    G0 = G_id(n0)

    # diagonal second derivatives
    for p in range(M):
        e = np.zeros(M, dtype=DT); e[p] = DT("1.0")
        Gp = G_id(n0 + h*e)
        Gm = G_id(n0 - h*e)
        H[p, p] = (Gp - DT("2.0") * G0 + Gm) / (h * h)

    # off-diagonal mixed partials
    for p in range(M):
        ep = np.zeros(M, dtype=DT); ep[p] = DT("1.0")
        for q in range(p+1, M):
            eq = np.zeros(M, dtype=DT); eq[q] = DT("1.0")
            Gpp = G_id(n0 + h*ep + h*eq)
            Gpm = G_id(n0 + h*ep - h*eq)
            Gmp = G_id(n0 - h*ep + h*eq)
            Gmm = G_id(n0 - h*ep - h*eq)
            val = (Gpp - Gpm - Gmp + Gmm) / (DT("4.0") * h * h)
            H[p, q] = val
            H[q, p] = val

    return H

def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT))


# =========================
# RUN VERIFICATION
# =========================
if __name__ == "__main__":
    print("Quadruplets (index order):")
    for idx, name in enumerate(quad_names):
        print(f"  {idx:2d}: {name}")

    print("\nBase n0 =", n0)
    print("Base G^id(n0) =", G_id(n0))

    Ha = H_id_analytic(n0)
    print("\n||H_analytic||_F =", float(fro_norm(Ha)))

    for h in H_LIST:
        if np.min(n0) <= h:
            print(f"\nSkipping h={h:g} (would make some n negative).")
            continue

        Hfd = H_id_fd(n0, h)

        rel_err = fro_norm(Hfd - Ha) / max(DT("1.0"), fro_norm(Ha))
        sym_err = fro_norm(Hfd - Hfd.T) / max(DT("1.0"), fro_norm(Hfd))

        print(f"\nh = {float(h):g}   relative error  = {float(rel_err):.3e}")
        #print(f"")
        #print(f"  symmetry error  = {sym_err:.3e}")

Quadruplets (index order):
   0: AA/XX
   1: AA/XY
   2: AA/YY
   3: AB/XX
   4: AB/XY
   5: AB/YY
   6: BB/XX
   7: BB/XY
   8: BB/YY

Base n0 = [1.  0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.2]
Base G^id(n0) = -55649.588580612108057

||H_analytic||_F = 45529.795945745595

h = 0.01   relative error  = 3.844e-04

h = 0.001   relative error  = 3.840e-06

h = 0.0001   relative error  = 3.842e-08

h = 1e-05   relative error  = 1.851e-08

h = 1e-06   relative error  = 2.932e-06
